# WB3240 - Systems and Control Engineering
- Write your answers to the questions in a separate report (see BrightSpace for more info); you may write your report in Dutch or English.

- Use the lecture slides, the course book, the (video) lectures, and the information on BrightSpace when solving the problems.

- Refer to the [Python Control Systems Library page](https://python-control.readthedocs.io/en/latest/) for function reference documentation. Also use the [Python-Control Cheat Sheet](https://c-school.org/wp-content/uploads/2021/09/python-control_cheat_sheet.pdf)!

- In some of the questions it is asked to draw a block diagram. A convenient and freely available online tool for this is https://draw.io.

- If you find any errors or typos in this document, please post this on the BrightSpace Discussion Board!

In [ ]:
%matplotlib inline 

import helper_functions
helper_functions.check_kernel()
helper_functions.check_control()

import numpy as np
import control as ct
import matplotlib.pyplot as plt

# change this value (default 140) if the plots are too large/small for your screen
plt.rcParams['figure.dpi'] = 140

from plot_tools import plot_system_response
from helper_functions import print_rounded

# This dummy constant is used to `wrongly` initialize variables that you have to find the right answer for
CHANGEME = None

Checking your Python kernel version.
All good! You have Python kernel version 3.10.
Checking the version of python-control.


## Block A, Part 4: Observers, Linear Quadratic Regulator (LQR)  | Crane control

Until now, we have assumed that the state-feedback controller has access to the full system state $x$. However, in practical scenarios, often only partial measurements of the state are available. In such cases, a state observer can estimate the full state from limited measurements, the system input, and a dynamical system model. A necessary condition for estimating the full state is that the system is observable.

_The open-loop (uncontrolled) linear system you found in <u>Question 3c</u> is taken as a starting point for this question. Make sure to copy-paste your $A$ and $B$ matrices below before continuing._

In [3]:
# System Parameters.
mc = 1.0e5  # Mass of the crane (kg)
mt = 6.3e5  # Mass of the tower (kg)
b = 1e5     # Translational friction of crane (Ns/m)
l = 110/2   # The tower is assumed to be a point mass at half the tower height (kg)
I = mt*l**2 # Inertia of the tower (kg m^2)
g = 9.81    # Gravitational constant (m/s^2)

# Names of the state-, input- and output-signals
states = ["theta_dot", "theta", "q_dot", "q"]
inputs = ["F"]
outputs = ["theta", "q"]
idx_theta_dot, idx_theta, idx_q_dot, idx_q = 0, 1, 2, 3

# Desired closed-loop eigenvalue locations
desired_cl_eigenvalues = np.array([-0.1982+0.4044j, -0.1982-0.4044j, -0.1223+0.1362j, -0.1223-0.1362j])


################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###


# Define the A and B matrices of the tower-on-crane system.
denominator2 = mt * mc * l **2 + I *( mt + mc )
num_a1 = -(1* mt * g *( mt + mc ) )
num_a2 = b *1* mt
num_a3 = g *1**2* mt **2
num_a4 = -( b * mt *1**2 + I * b )
num_b1 = -(1* mt )
num_b2 = mt *1**2+ I

A = np.array([
    [0, num_a1/denominator2, num_a2/denominator2, 0],
    [1, 0, 0, 0],
    [0, num_a3/denominator2, num_a4/denominator2, 0],
    [0, 0, 1, 0 ]])
B = np.array([
    [num_b1/denominator2],
    [0],
    [num_b2/denominator2],
    [0]])



###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################


C = np.array([[0, 1, 0, 0], [0, 0, 0, 1]])
D = np.array([[0], [0]])

n = A.shape[0]
p = B.shape[1]
q = C.shape[0]

sys = ct.ss(A, B, C, D, states=states, inputs=inputs, outputs=outputs)

## Observability

### 4a
Provide the symbolic definition of the observability matrix $W_\mathrm{o}$ for the considered system, only consisting of $A$, $C$, and products of both quantities. What is the maximum exponent of $A$, and how does this number relate to the order $n$ of the system? What quantities (state elements) do we measure? And what are, therefore, the dimensions of $W_\mathrm{o}$? How do your findings in this question relate the reachability matrix $W_\mathrm{r}$? Provide an answer in your report that addresses the questions asked. 












### 4b
Define $W_\mathrm{o}$ in the cell below (you may use already existing variables) and determine the rank of the matrix (use the function [**np.linalg.matrix_rank**](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html)). What can you conclude on the observability of the system? Explain what this conclusion means in practice for being able to reconstruct the full state of the considered system.<br><br>
_Hint: you can check your answer with the built-in Python Control functions [**ct.obsv**](https://python-control.readthedocs.io/en/latest/generated/control.obsv.html) and [**np.linalg.matrix_rank**](https://numpy.org/doc/stable/reference/generated/numpy.linalg.matrix_rank.html)_












In [4]:

################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###


# Determining observability from the available system measurements
W_r = np.vstack([C,C@A,C@A@A,C@A@A@A])
rank = np.linalg.matrix_rank(W_r)



###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################


print(f"The rank of the observability matrix: {rank}")

The rank of the observability matrix: 4


### 4c
What can you say about the observability of the system when only measuring the crane position $q$? And what if we only measure the tower angle $\theta$? What do you conclude? Can we still reconstruct the full state in both cases? Give a physical interpretation for both cases (i.e., why would/wouldn't it be possible to reconstruct the full state based on these single measurements?).

_You can use the code cell below._












In [7]:

################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###


# Determine the observability when only measuring the crane position, and when only measuring the tower angle.
C_q = np.array([0,0,0,1])
C_theta = np.array([0,1,0,0])
Wo_q = np.vstack([C_q,C_q@A,C_q@A@A,C_q@A@A@A])
Wo_theta = np.vstack([C_theta,C_theta@A,C_theta@A@A,C_theta@A@A@A])
rank_q = np.linalg.matrix_rank(Wo_q)
rank_theta = np.linalg.matrix_rank(Wo_theta)

print("rank_q = Wo_q", rank_q)
print("rank_theta = Wo_thetha", rank_theta)



###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################



rank_q = Wo_q 4
rank_theta = Wo_thetha 3


## Observer design and state estimation
We are now about to design the observer. First, we will analytically derive the full state-space representation of the system and the observer (without a state-feedback controller, $K=0$). The systems used in this question are defined as:
\begin{equation*}
\begin{aligned}
    \dot{x} &= Ax+Bu\\
    y &= Cx+Du\\
    \dot{\hat{x}} &= A\hat{x}+Bu + L(y-C\hat{x})\\
\end{aligned}
\begin{aligned}
&\left.\vphantom{\begin{aligned}
    \dot{x} &= Ax+Bu\\
    y &= Cx+Du
  \end{aligned}}\right\rbrace\quad\text{System}\\
&\left.\vphantom{\begin{aligned}
    \dot{\hat{x}} &= A\hat{x}+Bu + L(y_\mathrm{o}-C\hat{x})\\
  \end{aligned}}\right\rbrace\quad\text{Observer}\\
\end{aligned}
\end{equation*}
with $L$ being the observer gain, and $\hat{x}$ the estimated state.

**We assume in all the following questions we are measuring $\theta$ and $q$ unless stated differently.**

### 4d
A description of the full state-feedback controlled and observed system is given in the book (Eq. 8.18, PDF-page 276, book v3.1.5). However, in this exercise, we will design a state observer for the open-loop uncontrolled system without a state feedback controller. For the uncontrolled system, provide a mathematical derivation that leads to a state-space representation for our situation (Similar to Eq. 8.18). Include the derivation and resulting state-space system in your report, assuming that the direct feedthrough matrix $D=0$. Use the notation $A_\mathrm{e}$, $B_\mathrm{e}$, $C_\mathrm{e}$ (, and $D_\mathrm{e}$?) for the state-space system matrices. _Note: defining the state vector as either $[x^\top,\,\tilde{x}^\top]^\top$ or $[x^\top,\,\hat{x}^\top]^\top$ is correct._

Furthermore, provide the block diagram that represents the state-space system. Your block diagram should include the process (plant/system) and observer. Clearly indicate in your block diagram: $A$, $B$, $C$, $L$, an integrator blocks, and the signals $\dot{x}$, $x$, $\dot{\hat{x}}$, $\hat{x}$, $u$, $y$, $\hat{y}$ and $e = y-\hat{y}$.












-----
_Since the state-feedback controller operates based on the estimated state $\hat{x}$ in a closed-loop system, the observer should be faster than the controller. This is achieved by placing the observer's eigenvalues farther left in the complex plane than those of the closed-loop system. A common rule of thumb is to set the observer’s eigenvalues $4$ to $10$ times faster than the slowest eigenvalue of the controlled system. However, excessively fast observer dynamics can amplify noise and measurement errors, potentially leading to instability or degraded performance._

### 4e
Determine the dominant eigenvalue (pair?) from `desired_cl_eigenvalues` defined at the start of this assignment. Next, use the [**ct.place**](https://python-control.readthedocs.io/en/latest/generated/control.place.html) command to compute the observer gain $L$, placing the observer eigenvalues at locations whose magnitudes are $8\times$ larger than the dominant eigenvalue(s). Use `L` to define the state-space system you found in <u>Question 4d</u> in the cell below. The system outputs $(\theta,q)$ remain the same as in previous questions. 

In your report:
* Provide the resulting matrix in $L$ in your report. What is the dimension of $L$?
* Provide the <u>dimensions</u> of the resulting $A_\mathrm{e}$, $B_\mathrm{e}$, $C_\mathrm{e}$, $D_\mathrm{e}$ matrices of the open-loop system with state observer found in a previous question. Give symbolic (in terms of $n$, $p$, and $q$) and numeric expressions of the matrix dimensions, based on the dimensions of the original system. Explain why these matrices have these dimensions.

_Note: Because the state vector is extended with terms $\tilde{x}$, some matrices must be augmented with zeros._<br>
_Hint: To calculate the observer gain, the [**ct.place**](https://python-control.readthedocs.io/en/latest/generated/control.place.html) function must be called with the transposed matrices $A^\top$ and $C^\top$; its output must then be transposed to obtain $L$._












In [9]:

################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###


# Use `desired_cl_eigenvalues`, which was previously given, as a starting point.
# Your final answer is the Ae, Be, Ce, De matrices of the open-loop system + state observer found in a previous question
#given desired eigenvalues
desired_cl_eigenvalues = np.array([-0.1982+0.4044j, -0.1982-0.4044j, -0.1223+0.1362j, -0.1223-0.1362j])

#observer eigenvalues scaling
observer_eigenvalues = desired_cl_eigenvalues * 8

#observer gain L
L = ct.place(A.T, C.T, observer_eigenvalues).T

#augmented matrices
Ae = np.block([[A,np.zeros((4,4))],
[L@C,A-L@C]])
Be = np.vstack([B,B])
Ce = np.hstack([C, np.zeros((2,4))])
De = np.zeros((2,1))

#printed resulst 
print (" Observer gain L:\n", L )
print ("\n Augmented Matrices :\n")
print ("Ae :\n", Ae )
print ("Be :\n", Be )
print ("Ce :\n", Ce )
print ("De :\n", De )





###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################



 Observer gain L:
 [[ 5.255 -1.415]
 [ 2.584 -2.147]
 [ 1.201  4.610]
 [ 2.141  2.424]]

 Augmented Matrices :

Ae :
 [[ 0.000 -0.003  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 1.000  0.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  0.002 -0.121  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  0.000  1.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  5.255  0.000 -1.415  0.000 -5.258  0.000  1.415]
 [ 0.000  2.584  0.000 -2.147  1.000 -2.584  0.000  2.147]
 [ 0.000  1.201  0.000  4.610  0.000 -1.199 -0.121 -4.610]
 [ 0.000  2.141  0.000  2.424  0.000 -2.141  1.000 -2.424]]
Be :
 [[-0.000]
 [ 0.000]
 [ 0.000]
 [ 0.000]
 [-0.000]
 [ 0.000]
 [ 0.000]
 [ 0.000]]
Ce :
 [[ 0.000  1.000  0.000  0.000  0.000  0.000  0.000  0.000]
 [ 0.000  0.000  0.000  1.000  0.000  0.000  0.000  0.000]]
De :
 [[ 0.000]
 [ 0.000]]


### 4f
Use the code cell below to demonstrate that your observer is able to estimate the full state vector $x$. Perform a simulation in which the cart is actuated for a time interval $T$ with force $F_1$, followed by applying the opposite force $-F_1$ for the same duration. This ensures the cart moves and then (approximately) returns to its original position.

In your report:
* Include the code used to perform the simulation described above.
* Show simulation results in a plot where you display the actual (true) states $x$ and compare them with the estimated states $\hat{x}$;
* Vary the observer eigenvalue locations and comment on how it affects the performance of the observer in simulation (show in the plot);

_Hint: If you initialize the system state and the observer state with the same values, their trajectories may look nearly identical. To better demonstrate the observer's ability to converge to the true state, consider starting the observer with an initial estimate that differs from the initial system state. You can also experiment with reducing the magnitude of the observer poles to evaluate how this affects the speed and quality of the state estimation._

_Note: This exercise is intentionally challenging. You are expected to design and implement the full simulation code yourself, including the system dynamics, observer, input sequence, and plotting._












In [ ]:

################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###
F1 = 2000000 #N
Ts = 100
dt = 0.01
T = np.arange(0, 2*Ts + dt
 





###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################



# State estimation with noisy measurements and numerical differentiation
In this exercise, we explore why numerical differentiation performs poorly when measurements contain noise and why state observers are the preferred solution for reconstructing the full state vector in real systems. The numerical differentiation approach is common in practical control systems when only position‑type variables are measured, but it becomes unreliable as soon as noise is present.

We consider the same 4th‑order state‑space model used earlier. Only two states are measured: the pendulum angle $\theta$ and the cart position $q$, whereas the velocity states $\dot{\theta}$ and $\dot{q}$ are not measured. We will attempt to reconstruct them using numerical differentiation

### 4g
First, simulate the nominal system (without observer) for $T_\mathrm{Max} = 50$ seconds with a time step of $\Delta t = 0.01$ seconds, using the input
\begin{aligned}
u(t)=10^6\sin⁡{(0.25\pi t)}.
\end{aligned}
In practice, sensor measurements are corrupted by noise. We model noisy system output measurements as
\begin{aligned}
    y(t)&=Cx(t)+v(t),
\end{aligned}
In this exercise, you will add such noise $v(t)$ to the measured angle and position signals (simulation output) with a sensible variance, and then attempt to recover the velocity states by differentiating the noisy measurements. 

After simulating the system and adding noise, use numerical differentiation (for example, a backward difference) to compute estimates of $\dot{\theta}(t)$ and $\dot{q}(t)$. You should then compare these numerically differentiated signals with the true velocity states obtained directly from the system simulation. Present both the true and estimated velocities together in clear plots so that the effect of the noise is immediately visible. Finally, reflect on why numerical differentiation performs so poorly in this situation, and explain in your own words why differentiation inherently amplifies high‑frequency components such as measurement noise.

Include and describe in your report: 
1. A brief description of how you added zero‑mean Gaussian noise to $\theta(t)$ and $q(t)$.
1. Your numerical differentiation method.
1. Plots comparing true and estimated (differentiated) velocities for both states.
1. A short explanation of why numerical differentiation amplifies measurement noise.












In [ ]:
### Simulation setup
DT = 0.01
TMax = 50
T  = np.arange(0, TMax, DT)
N  = len(T)

# Input
u = 1e6*np.sin(0.25*np.pi*T)

# Initial condition
x0 = np.zeros(A.shape[0])


################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###






###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################



## Linear Quadratic Regulator - Optimal controller design
In the previous question, you designed a state observer for the tower-in-crane system. In this exercise, we will extend the state-observed system by incorporating a state-feedback controller using linear quadratic regulator (LQR). This is possible as the _separation principle_ states that if a system is both controllable and observable, we can design a stable closed-loop system by independently designing a state observer and a state-feedback controller, and then combining them.

Previously (Part 3), you computed the state-feedback gain $K$ using manual pole placement. However, designing state-feedback controllers for higher-order systems becomes increasingly complex. Therefore, in this question, we will use LQR-based controller design.

### 4h
We ask you to design a full-state feedback controller using LQR controller design. The state observer should act on the observed state $\hat{x}$ and has the form $u = -K\hat{x} + k_\mathrm{f}r$. Design multiple (2 or more) LQR controller designs that demonstrate different trade-offs between state and control input penalization. To evaluate your LQR closed-loop controlled system, show unit step responses that bring the crane position $q$ to $1$ m.

In your report, we expect you to: 
* Include and comment on the code leading to your results.
* Motivate your choices for the weights `Qx` and `Qu` for each of the cases, and present (closed-loop) simulation results in which you plot the output $\theta$ and $q$, and the (scaled) controller input. 
* Relate your step response observations to the choices you made for the different LQR controller designs.
* Optionally, provide the resulting closed-loop eigenvalues. Explain your observations.

_Note and hints_: 
* Refer to the [**ct.lqr**](https://python-control.readthedocs.io/en/latest/generated/control.lqr.html) documentation page of the Python Control System Library. The optional argument `method='scipy'` for the [**ct.lqr**](https://python-control.readthedocs.io/en/latest/generated/control.lqr.html) function might solve a common problem using the function in the Vocareum environment.
* The state and input penalization terms in the infinite-horizon cost function $J$ must be in the same order of magnitude for the optimization problem to be well-defined. To attain this, scale the control (input) matrix $B$ with a (large!) constant factor, or use a very small value for $Q_\mathrm{x}$.
* You need a distinct reference gain $k_\mathrm{f}$ for each of the closed-loop systems to correctly follow the reference you apply to the system.

_Note: This exercise is intentionally challenging and requires you to design and analyze several controllers designed with LQR. It is a required part of the assignment (not a bonus exercise)._












In [ ]:

################################################################################
###              ↓↓↓   Start writing your own code here:   ↓↓↓               ###


# Create the closed-loop system based on estimated state feedback here, 
# design different state-feedback controllers using LQR (different state and input weights),
# and simulate the closed-loop responses to a step command.




###              ↑↑↑   Stop writing your own code here.    ↑↑↑               ###
################################################################################



### End of Block A ✅